# Client Lifetime Value (LTV) Forecasting

This notebook demonstrates a Client Lifetime Value (LTV) forecasting model. The goal is to predict the commission each client will generate in the next window, allowing an organization to prioritize client coverage and service levels.

## 1. Setup and Data Loading

This section imports necessary libraries and loads the raw transaction data. It also performs initial data type conversions and splits the data into historical and future periods based on a defined cutoff date.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Load the dataset from the Excel file
df = pd.read_excel("Financial Dataset.xlsx", sheet_name="Sheet1")

# Convert 'Trading Date' column to datetime objects
df["Trading Date"] = pd.to_datetime(df["Trading Date"])

# Define the split date for historical vs. target window
split = pd.Timestamp("2023-04-01")

# Separate data into historical and future periods
hist = df[df["Trading Date"] < split]
fut = df[df["Trading Date"] >= split]

print("Data loaded and split into historical and future periods.")

Data loaded and split into historical and future periods.


## 2. Feature Engineering

Here, we calculate historical features for each client based on their past trading activity. These features include total turnover, total commission, last reported equity, total deposits, and the number of active trading days.

In [3]:
# Aggregate historical features per client
feat = hist.groupby("Customer ID").agg(
    turnover_h = ("Turnover", "sum"),
    comm_h = ("Total Commission", "sum"),
    equity_h = ("Equity", "last"),
    deposits_h = ("Deposit", "sum"),
    active_days = ("Trading Date", "nunique"),
).reset_index()

print("Historical features engineered.")
display(feat.head())

Historical features engineered.


,Customer ID,turnover_h,comm_h,equity_h,deposits_h,active_days
0,C1,8794209.1,35176.85,3.187620e+04,281984000.0,38
1,C10,19304966.3,57914.93,1.517867e+06,645634000.0,52
2,C100,97018.5,339.56,3.724600e+05,2591000.0,6
3,C1000,2980.0,11.92,1.106047e+06,1089000.0,1
4,C1001,4016833.5,16067.33,1.083064e+06,10000000.0,10


## 3. Target Variable Definition and Data Preparation

This section defines the target variable, which is the total commission generated by each client in the future window. It then merges the historical features with this target and prepares the data for model training by handling missing values and splitting into features (X) and target (y).

In [4]:
# Calculate the target: total commission in the future window per client
target = (fut.groupby("Customer ID")["Total Commission"]
          .sum().rename("comm_next"))

# Merge historical features with the target variable
data = feat.merge(target, on="Customer ID", how="left")

# Fill any NaN values in 'comm_next' with 0 (for clients with no future commission)
data["comm_next"] = data["comm_next"].fillna(0)

# Define feature matrix (X) and target vector (y)
X = data[["turnover_h", "comm_h", "equity_h", "deposits_h", "active_days"]]
y = data["comm_next"]

# Split data into training and testing sets
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)

print("Target variable defined and data prepared for modeling.")
display(data.head())

Target variable defined and data prepared for modeling.


,Customer ID,turnover_h,comm_h,equity_h,deposits_h,active_days,comm_next
0,C1,8794209.1,35176.85,3.187620e+04,281984000.0,38,4779.64
1,C10,19304966.3,57914.93,1.517867e+06,645634000.0,52,33308.35
2,C100,97018.5,339.56,3.724600e+05,2591000.0,6,2779.50
3,C1000,2980.0,11.92,1.106047e+06,1089000.0,1,2719.10
4,C1001,4016833.5,16067.33,1.083064e+06,10000000.0,10,16377.20


## 4. Model Training and Evaluation

A Gradient Boosting Regressor model is initialized and trained on the historical data. After training, the model's performance is evaluated using Mean Absolute Error (MAE) and R-squared (R2) on the test set.

In [5]:
# Initialize and train the Gradient Boosting Regressor model
reg = GradientBoostingRegressor(
    n_estimators=400, learning_rate=0.05,
    max_depth=3, random_state=42)
reg.fit(Xtr, ytr)

# Make predictions on the test set
pred = reg.predict(Xte)

# Evaluate the model's performance
print("MAE:", round(mean_absolute_error(yte, pred), 2))
print("R2 :", round(r2_score(yte, pred), 3))

print("Model trained and evaluated.")

MAE: 18625.07
R2 : 0.382
Model trained and evaluated.


## 5. Client Scoring and Tiering

Finally, the trained model is used to forecast the LTV for all clients. Clients are then categorized into 'Bronze', 'Silver', 'Gold', and 'Platinum' tiers based on quartiles of their predicted LTV. The results are saved to a CSV file, and a summary of tier distribution is printed.

In [6]:
# Predict LTV for all clients in the dataset
data["ltv_forecast"] = reg.predict(X)

# Create client tiers based on LTV forecast quartiles
data["tier"] = pd.qcut(data["ltv_forecast"], 4,
                          labels=["Bronze", "Silver", "Gold", "Platinum"])

# Sort clients by LTV forecast and save to a CSV file
data.sort_values("ltv_forecast", ascending=False).to_csv(
    "ltv_tiers.csv", index=False)

# Print a summary of LTV forecasts by tier
print("LTV forecast generated and clients tiered:")
print(data.groupby("tier")["ltv_forecast"].agg(["count", "mean"]))

LTV forecast generated and clients tiered:
          count          mean
tier                         
Bronze      407    798.462078
Silver      407   3273.089634
Gold        408  11615.858151
Platinum    406  77097.638979


/tmp/ipykernel_1145/185461819.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(data.groupby("tier")["ltv_forecast"].agg(["count", "mean"]))


## Interpretation and Actionable Insights

The Client Lifetime Value (LTV) forecasting model has successfully categorized clients into four tiers: Bronze, Silver, Gold, and Platinum, based on their predicted future commission. The summary output shows the count of clients in each tier and their average forecasted LTV.

*   **Bronze Tier:** These clients have the lowest forecasted LTV, with an average LTV of approximately \$798. They represent the largest segment in terms of client count. While their individual LTV is low, collectively they could still contribute a significant amount.
*   **Silver Tier:** Clients in this tier have a moderate forecasted LTV, averaging around \$3,273. This segment also has a substantial number of clients.
*   **Gold Tier:** These clients show a higher forecasted LTV, averaging approximately \$11,616. They are fewer in number than Bronze and Silver, but contribute significantly more per client.
*   **Platinum Tier:** This is the smallest but most valuable segment, with the highest forecasted LTV, averaging around \$77,097 per client. These are the top-tier clients that drive a substantial portion of future commission.

### Actionable Strategies for Financial Organizations:

1.  **Resource Allocation for Relationship Managers (RMs):**
    *   **Platinum Clients:** Allocate the most experienced and dedicated RMs to these clients. Focus on white-glove service, personalized investment advice, and proactive engagement to maximize retention and identify cross-selling opportunities.
    *   **Gold Clients:** Assign skilled RMs to maintain strong relationships. Implement strategies to deepen engagement, offer tailored products, and encourage increased trading activity to potentially elevate them to Platinum.
    *   **Silver Clients:** RMs should engage regularly but perhaps with a more standardized approach. Focus on educational content, value-added services, and opportunities for account growth. Consider automated touchpoints for efficiency.
    *   **Bronze Clients:** Utilize digital channels and automated communications for cost-effective engagement. Offer self-service options and targeted promotions to encourage increased activity or product adoption. RMs might only engage on an as-needed basis or for specific campaigns.

2.  **Targeted Marketing and Product Development:**
    *   **Platinum/Gold:** Develop exclusive products, premium services, or loyalty programs to reward and retain these high-value clients.
    *   **Silver/Bronze:** Design acquisition or upgrade campaigns focused on moving clients up the tiers. For instance, offer incentives for increased trading volume or new product subscriptions.

3.  **Churn Prevention:**
    *   Monitor activity for Gold and Platinum clients closely. Any drop in activity or engagement should trigger an immediate proactive outreach from RMs.
    *   For Silver and Bronze clients, use automated alerts for significant changes in behavior that might indicate churn risk.

4.  **Operational Efficiency:**
    *   By understanding the value distribution, a financial organization can optimize operational costs by automating services for lower-tier clients while investing more in personalized services for higher-tier clients.